# Baseline CVAE Test

In [1]:
from pathlib import Path
import pickle

import torch

from training import train_model
from model import PeptideCVAE
from inference import generate_peptides

In [2]:
# Paths relative to this notebook
baseline_dir = Path(".")
dataset_file = "../database/test.json"
model_path = baseline_dir / "cvae_peptide_model.pth"
scaler_path = baseline_dir / "scaler.pkl"

# Quick 1-epoch train
model, device = train_model(
    dataset_file=str(dataset_file),
    epochs=50,
    batch_size=32,
    max_len=12,
    latent_dim=32,
    model_path=str(model_path),
)

print(f"Model saved at: {model_path}")
print(f"Scaler expected at: {scaler_path}")

Using device: cuda
Scaler saved to scaler.pkl
Epoch [1/50] | Beta: 0.00 | Loss: 0.6810 | Recon: 0.6810 | KL: 1.6735
Epoch [5/50] | Beta: 0.16 | Loss: 0.4456 | Recon: 0.3898 | KL: 0.3485
Epoch [10/50] | Beta: 0.36 | Loss: 0.3877 | Recon: 0.3012 | KL: 0.2403
Epoch [15/50] | Beta: 0.56 | Loss: 0.3773 | Recon: 0.2814 | KL: 0.1713
Epoch [20/50] | Beta: 0.76 | Loss: 0.3672 | Recon: 0.2776 | KL: 0.1179
Epoch [25/50] | Beta: 0.96 | Loss: 0.3505 | Recon: 0.2801 | KL: 0.0734
Epoch [30/50] | Beta: 1.00 | Loss: 0.3200 | Recon: 0.2711 | KL: 0.0489
Epoch [35/50] | Beta: 1.00 | Loss: 0.2892 | Recon: 0.2533 | KL: 0.0358
Epoch [40/50] | Beta: 1.00 | Loss: 0.2627 | Recon: 0.2347 | KL: 0.0281
Epoch [45/50] | Beta: 1.00 | Loss: 0.2403 | Recon: 0.2181 | KL: 0.0222
Epoch [50/50] | Beta: 1.00 | Loss: 0.2205 | Recon: 0.2019 | KL: 0.0187
Model saved to 'cvae_peptide_model.pth'
Model saved at: cvae_peptide_model.pth
Scaler expected at: scaler.pkl


In [ ]:
# Reload model + scaler and generate a few peptides
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

# Keep architecture args aligned with training defaults
loaded_model = PeptideCVAE(
    vocab_size=24,
    condition_dim=8,
    max_seq_len=14,
    latent_dim=32,
)
loaded_model.load_state_dict(torch.load(model_path, map_location=device))

# Trying to imitate this:
# {
#     "dbaasp_id": "DBAASPS_373",
#     "sequence": "KLFKRWKHLFR",
#     "length": 11,
#     "smiles": "CC(C)C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N)CCCCN)C(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](CCCN=C(N)N)C(=O)O",
#     "ph": null,
#     "molecular_weight": 1558.9480000000003,
#     "logp": -0.992100000000006,
#     "net_charge": 5.0,
#     "isoelectric_point": 12.18,
#     "hydrophobicity": 1.05,
#     "cathionicity": 6,
#     "target_groups": [
#         "Gram+"
#     ],
#     "complexity": "Monomer"
# }
target = [11, 7, 1558.94, -0.9921, 5.0, 12.18, 1.05, 6]

samples = generate_peptides(
    model=loaded_model,
    scaler=scaler,
    num_samples=5,
    properties=target,
    temperature=0.9,
    top_k=5,
)

samples

Using device: cuda
Model loaded successfully.

Target Properties:
- Length: 11.0
- pH: 7.0
- Molecular Weight: 1567.99
- LogP: -0.31
- Net Charge: 6.0
- Isoelectric Point: 14.0
- Hydrophobicity: 0.51
- Cathionicity: 5.0

--- Generating 5 novel peptide(s) ---
Sample 1: WLLKWKRLLRK
Sample 2: RLLKRWKRLVL
Sample 3: RLLKRLLKWKW
Sample 4: RLLKRLKWLWK
Sample 5: FKWRRAIKIFK


['WLLKWKRLLRK', 'RLLKRWKRLVL', 'RLLKRLLKWKW', 'RLLKRLKWLWK', 'FKWRRAIKIFK']